# Winner to Steal — Steel Energy Forecasting

Final three-model ensemble: CatBoost + XGBoost + change-target CatBoost.

Validated final chronological holdout: **RMSE 8.116 kWh**, **MAE 3.843 kWh**.

In [ ]:
# Run this first in a new Colab session. No Google Drive is used.
%pip -q install "scikit-learn==1.6.1" catboost xgboost joblib pandas numpy

from google.colab import files

from pathlib import Path
import json
import time
import zipfile
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

print('Upload Winner_to_Steal_PC.zip and train.csv when prompted.')
uploaded_files = files.upload()

CONTENT_PATH = Path('/content')
ZIP_PATH = CONTENT_PATH / 'Winner_to_Steal_PC.zip'
TRAIN_PATH = CONTENT_PATH / 'train.csv'
PACKAGE_PATH = Path('/content/winner_to_steal_model')

if not ZIP_PATH.exists():
    raise FileNotFoundError('Upload Winner_to_Steal_PC.zip before continuing.')
if not TRAIN_PATH.exists():
    raise FileNotFoundError('Upload train.csv before continuing.')
print('Files uploaded successfully.')

In [ ]:
# Optional: inspect the training data.
df = pd.read_csv(TRAIN_PATH)
print('Training rows:', len(df))
display(df.head())

In [ ]:
# Extract and load the saved final model package.
PACKAGE_PATH.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_file:
    zip_file.extractall(PACKAGE_PATH)

with open(PACKAGE_PATH / 'steel_best_model_metadata.json', 'r') as file:
    metadata = json.load(file)

FEATURES = metadata['features']
RAW_COLUMNS = [
    'date', 'Usage_kWh', 'Lagging_Current_Reactive.Power_kVarh',
    'Leading_Current_Reactive_Power_kVarh', 'CO2(tCO2)',
    'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor',
    'NSM', 'WeekStatus', 'Day_of_week', 'Load_Type'
]

direct_catboost_model = CatBoostRegressor()
direct_catboost_model.load_model(PACKAGE_PATH / 'steel_catboost_energy.cbm')

xgboost_model = XGBRegressor()
xgboost_model.load_model(PACKAGE_PATH / 'steel_xgboost_energy.json')

change_catboost_model = CatBoostRegressor()
change_catboost_model.load_model(PACKAGE_PATH / 'steel_change_catboost_energy.cbm')

preprocessor = joblib.load(PACKAGE_PATH / 'steel_energy_preprocessor.joblib')
print('Winner to Steal model loaded successfully.')

In [ ]:
def forecast_winner(history_df, future_df):
    history = history_df[RAW_COLUMNS].tail(672).copy()
    history['_source'] = 'history'
    future = future_df[RAW_COLUMNS].copy()
    future['_source'] = 'future'
    data = pd.concat([history, future], ignore_index=True)
    data['date'] = pd.to_datetime(data['date'], format='mixed', dayfirst=True)
    data = data.sort_values('date').reset_index(drop=True)

    data['hour'] = data['date'].dt.hour
    data['minute'] = data['date'].dt.minute
    data['month'] = data['date'].dt.month
    data['day_number'] = data['date'].dt.dayofweek
    data['is_weekend'] = (data['day_number'] >= 5).astype(int)
    data['time_sin'] = np.sin(2 * np.pi * data['NSM'] / 86400)
    data['time_cos'] = np.cos(2 * np.pi * data['NSM'] / 86400)
    data['reactive_power_total'] = (
        data['Lagging_Current_Reactive.Power_kVarh']
        + data['Leading_Current_Reactive_Power_kVarh']
    )
    data['power_factor_difference'] = (
        data['Lagging_Current_Power_Factor']
        - data['Leading_Current_Power_Factor']
    )
    for lag in [1, 2, 4, 8, 16, 24, 96, 192, 672]:
        data[f'usage_lag_{lag}'] = data['Usage_kWh'].shift(lag)
    for window in [4, 16, 96, 672]:
        data[f'usage_mean_{window}'] = data['Usage_kWh'].rolling(window).mean()
        data[f'usage_std_{window}'] = data['Usage_kWh'].rolling(window).std()
    data['usage_change_1'] = data['Usage_kWh'].diff(1)
    data['usage_change_4'] = data['Usage_kWh'].diff(4)

    rows = data.loc[data['_source'].eq('future')].copy()
    if rows[FEATURES].isnull().any().any():
        raise ValueError('Missing lag features. Supply at least 672 historical records.')

    cat_prediction = direct_catboost_model.predict(rows[FEATURES])
    xgb_prediction = xgboost_model.predict(preprocessor.transform(rows[FEATURES]))
    change_prediction = (
        rows['Usage_kWh'].to_numpy()
        + change_catboost_model.predict(rows[FEATURES])
    )
    prediction = (
        metadata['direct_catboost_weight'] * cat_prediction
        + metadata['xgboost_weight'] * xgb_prediction
        + metadata['change_catboost_weight'] * change_prediction
    )
    return pd.DataFrame({
        'forecast_for': rows['date'] + pd.Timedelta(minutes=15),
        'predicted_next_interval_Usage_kWh': prediction
    })

In [ ]:
# The official test.csv has not been released, so no prediction is run here.
# When it is released, upload it through the Files panel and then use this code:
# test_data = pd.read_csv('/content/test.csv')
# history_data = pd.read_csv(PACKAGE_PATH / 'steel_history_context.csv')
# submission = forecast_winner(history_data, test_data)
# submission.to_csv('/content/winner_to_steal_predictions.csv', index=False)

print('Model loaded and ready. No test.csv is required at this stage.')